In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

from utils import Vcom, get_comoving_distance_h, c, z_min
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
import copy

# from Ng8_func import Ng8_percentile_wprp, Ng8_percentile_wprp_theory, return_box_rsd_pos
from Ng8_func import N_g_8, N_g_8_theory, rand_z_assign, select_top20_in_random, select_bottom20_in_random, cross_correlation_wprp, cross_correlation_wprp_theory, return_box_rsd_pos

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_r: [-22,-21]$',r'$M_r: [-21,-20]$',r'$M_r: [-20,-19]$',r'$M_r: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

In [ ]:
M_thr_arr_DDP  = np.array((-20.0, -20.0, -19.0, -18.0))
z_max_arr_DDP  = np.array((31900, 31900, 19250, 12500))/ c

In [ ]:
### Comoving pi_max and wp bins in Mpc/h
rp_bins = np.logspace(-0.8, 1.6, 13)
rp_bins_x = (rp_bins[1:] + rp_bins[:-1]) / 2


from matplotlib import cm

colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]

In [ ]:
def Ng8_percentile_wprp_DDP(gal, random, Ng8_thr, pi_max, rp_bins, nthread = 55):
    
    D1_ra  = copy.deepcopy(gal[:, 0])
    D1_dec = copy.deepcopy(gal[:, 1])
    D1_z   = copy.deepcopy(gal[:, 2])
    R1_ra  = copy.deepcopy(random[:, 0])
    R1_dec = copy.deepcopy(random[:, 1])
    R1_z   = copy.deepcopy(random[:, 2])

    # len(Ng8_thr) = len(D1_ra)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)

    D2_ra  = select_bottom20_in_random(D1_ra , Ng8_thr)
    D2_dec = select_bottom20_in_random(D1_dec, Ng8_thr)
    D2_z   = select_bottom20_in_random(D1_z  , Ng8_thr)

    R2_ra  = R1_ra
    R2_dec = R1_dec
    R2_z   = rand_z_assign(R1_z, D2_z) # shuffle z from D2_z(thr masked)

    wprp_bottom20 = cross_correlation_wprp( D1_ra, D1_dec, D1_z, R1_ra, R1_dec, R1_z, 
                                            D2_ra, D2_dec, D2_z, R2_ra, R2_dec, R2_z,
                                            pi_max, rp_bins, gal_weights = 1.0, random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 top 20%)

    D2_ra  = select_top20_in_random(D1_ra , Ng8_thr)
    D2_dec = select_top20_in_random(D1_dec, Ng8_thr)
    D2_z   = select_top20_in_random(D1_z  , Ng8_thr)

    R2_ra  = R1_ra
    R2_dec = R1_dec
    R2_z   = rand_z_assign(R1_z, D2_z) # shuffle z from D2_z(thr masked)

    wprp_top20 = cross_correlation_wprp( D1_ra, D1_dec, D1_z, R1_ra, R1_dec, R1_z, 
                                            D2_ra, D2_dec, D2_z, R2_ra, R2_dec, R2_z,
                                            pi_max, rp_bins, gal_weights = 1.0, random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    return wprp_bottom20, wprp_top20


In [ ]:
def Ng8_percentile_wprp_theory_DDP(gal, random, Ng8_thr, pi_max, rp_bins, boxsize, nthread = 55):
    D1_X = copy.deepcopy(gal[:, 0])
    D1_Y = copy.deepcopy(gal[:, 1])
    D1_Z = copy.deepcopy(gal[:, 2])
    R1_X = copy.deepcopy(random[:, 0])
    R1_Y = copy.deepcopy(random[:, 1])
    R1_Z = copy.deepcopy(random[:, 2])

 
    # len(Ng8_thr) = len(D1_ra)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)

    D2_X = select_bottom20_in_random(D1_X, Ng8_thr)
    D2_Y = select_bottom20_in_random(D1_Y, Ng8_thr)
    D2_Z = select_bottom20_in_random(D1_Z, Ng8_thr)

    R2_X = R1_X
    R2_Y = R1_Y
    R2_Z = R1_Z

    wprp_bottom20 = cross_correlation_wprp_theory(  D1_X, D1_Y, D1_Z, R1_X, R1_Y, R1_Z, 
                                                    D2_X, D2_Y, D2_Z, R2_X, R2_Y, R2_Z,
                                                    pi_max, rp_bins, boxsize, gal_weights  = 1.0, 
                                                    random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 top 20%)

    D2_X = select_top20_in_random(D1_X, Ng8_thr)
    D2_Y = select_top20_in_random(D1_Y, Ng8_thr)
    D2_Z = select_top20_in_random(D1_Z, Ng8_thr)

    R2_X = R1_X
    R2_Y = R1_Y
    R2_Z = R1_Z

    wprp_top20 = cross_correlation_wprp_theory( D1_X, D1_Y, D1_Z, R1_X, R1_Y, R1_Z, 
                                                D2_X, D2_Y, D2_Z, R2_X, R2_Y, R2_Z,
                                                pi_max, rp_bins, boxsize, gal_weights = 1.0, 
                                                random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    return wprp_bottom20, wprp_top20

#### Mr_C_bins

In [ ]:
for j in range(4):
    print(M_thr_arr[j],M_thr_arr[j+1], '|', M_thr_arr_DDP[j],  '|', z_max_arr_DDP[j])


#### CCMD 32 quads & SDSS bright1

In [ ]:
data = fits.open('../CCMD_main_processed/CCMD_Main_Quad0_processed_catalog.fits')[1].data

data

In [ ]:
### CCMD 4 Quads
paths = ['Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',]

for path in paths:
    data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
    
    N_total = len(data) # 获取母表总长度

    output_dict1 = collections.defaultdict(list)
    output_dict2 = collections.defaultdict(list)
#########################################################################################################
    # for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
    for j in [0]: # Mr[-22,-21] sample Ng8 is almost 0
        ### DDP ###################################################################################
        mask_DDP = np.logical_and(  data['Mr'] < M_thr_arr_DDP[j],
                                    np.logical_and(z_min <= data['z'], data['z'] < z_max_arr_DDP[j] ) )
        data_DDP = data[mask_DDP]
        pos_DDP  = np.array([   np.array(data_DDP['ra']),
                                np.array(data_DDP['dec']),    
                                np.array(data_DDP['z'])]).T
        Ng8_DDP  = N_g_8(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

        # === 高效映射核心步骤 ===
        # 创建一个与原始 data 等长的数组，初始化为 NaN
        Ng8_full = np.full(N_total, np.nan)
        # 将算好的 Ng8 填入 DDP 星系所在的位置
        Ng8_full[mask_DDP] = Ng8_DDP
        # ========================
        ###########################################################################################
        for i in range(4):
            stat_bot = []
            stat_top = []
            ### Mr mask is related to the z_max 
            mask = np.logical_and(  np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                    np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                    np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
            if j == 3:
                pi_max = 40.
            else :
                pi_max = 60.

            data_thr = data[mask]
            # === 高效提取对应星系的 Ng8 ===
            # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
            Ng8_thr  = Ng8_full[mask]
            
            # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
            # 如果你的物理逻辑保证了 thr 样本一定是 DDP 的子集，这一步不会剔除任何星系
            valid_idx = ~np.isnan(Ng8_thr)
            data_thr_valid = data_thr[valid_idx]
            Ng8_thr_valid  = Ng8_thr[valid_idx]
            # ==============================

            coords_gal_thr  = np.array([np.array(data_thr_valid['ra']),
                                        np.array(data_thr_valid['dec']),    
                                        np.array(data_thr_valid['z'])]).T
            coords_rand_thr = np.load('../CCMD_mag_bin/Mr_wp_randoms/ccmd_' + path + '_randoms_shuffle_z_' + M_thr_list[j] + '.npz')['randoms']

            wprp_bottom20, wprp_top20 = Ng8_percentile_wprp_DDP(coords_gal_thr, coords_rand_thr, Ng8_thr_valid, pi_max, rp_bins, )
            stat_bot.append(wprp_bottom20)
            stat_top.append(wprp_top20)

            output_dict1['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_bot for item in sublist])
            output_dict2['wprp_top20_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_top for item in sublist])

    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_' + path + '_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz', **output_dict1)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_' + path + '_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz', **output_dict2)

#### SDSS bright1 jackknife cov

In [ ]:
# /share/wenlk/AssemblyBias/ABcode/code/CCMD_32Quads/CCMD_32Quads_Ng8_2pcf_Mr_C_Ng8in_Mr_thr_DDP.ipynb

#### CCMD Main 4 quads cov


In [ ]:
### CCMD main 4 quads
paths = ['Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',]

output_dict_bottom_stat = collections.defaultdict(list)
output_dict_top_stat    = collections.defaultdict(list)
output_dict_bottom_cov  = collections.defaultdict(list)
output_dict_top_cov     = collections.defaultdict(list)

# for j in range(4): # Mr, z
for j in [0]: # Mr, z
    for i in range(4): # Color
        bottom_data  = []
        top_data     = []
        for path in paths:
            bottom_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_' + path + '_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
            bottom_data.append(bottom_temp['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

            top_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_' + path + '_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')
            top_data.append(top_temp['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

        bottom_data  = np.array(bottom_data)
        top_data     = np.array(top_data)     

        bottom_stat = []
        bottom_cov  = []
        top_stat    = []
        top_cov     = []

        original_statistics_bottom = np.mean(bottom_data, axis=0)
        original_statistics_top    = np.mean(top_data    , axis=0)
        bottom_cov_matrix          = np.cov(bottom_data, rowvar=False)  # rowvar=False, column for variants
        top_cov_matrix             = np.cov(top_data, rowvar=False)  # rowvar=False, column for variants
        
        # bottom_cov_matrix          = np.std(bottom_data, axis=0)  # rowvar=False, column for variants
        # top_cov_matrix             = np.std(top_data, axis=0)  # rowvar=False, column for variants

        bottom_stat.append(original_statistics_bottom)
        bottom_cov.append(bottom_cov_matrix)
        top_stat.append(original_statistics_top)
        top_cov.append(top_cov_matrix)

        output_dict_bottom_stat['wprp_bottom20_'     + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in bottom_stat for item in sublist])
        output_dict_bottom_cov[ 'wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in bottom_cov for item in sublist])
        output_dict_top_stat['wprp_top20_'     + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in top_stat for item in sublist])
        output_dict_top_cov[ 'wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in top_cov for item in sublist])

    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_stat_mean.npz', **output_dict_bottom_stat)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_stat_mean.npz', **output_dict_top_stat)
    
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_cov.npz', **output_dict_bottom_cov)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_cov.npz', **output_dict_top_cov)

#### CCMD main box w/ rsd

In [ ]:
# /share/wenlk/AssemblyBias/ABcode/code/CCMD_32Quads/CCMD_32Quads_Ng8_2pcf_Mr_C_Ng8in_Mr_thr_DDP.ipynb

#### CCMD 20Vars box, rsd

In [ ]:
# /share/wenlk/AssemblyBias/ABcode/code/CCMD_32Quads/CCMD_32Quads_Ng8_2pcf_Mr_C_Ng8in_Mr_thr_DDP.ipynb

### Plot

In [ ]:
CCMD_Main_4Quads_bottom20     = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_stat_mean.npz')
CCMD_Main_4Quads_bottom20_cov = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_cov.npz')
CCMD_Main_4Quads_top20        = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_stat_mean.npz')
CCMD_Main_4Quads_top20_cov    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/Main_4Quads_Mrp22p21/CCMD_Main_4Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_cov.npz')

In [ ]:
SDSS_Mr_C_bottom20        = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
SDSS_Mr_C_top20           = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')
SDSS_cov_Mr_C_bottom20    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_cov_pimax60_60_60_40.npz')
SDSS_cov_Mr_C_top20       = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_cov_pimax60_60_60_40.npz')
# SDSS_cov_Mr_C_bottom20_jk = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_cov_jkknife_sample_pimax60_60_60_40.npz')
# SDSS_cov_Mr_C_top20_jk    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_cov_jkknife_sample_pimax60_60_60_40.npz')

CCMD_32Quads_bottom20     = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_stat_mean.npz')
CCMD_32Quads_bottom20_cov = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_cov.npz')
CCMD_32Quads_top20        = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_stat_mean.npz')
CCMD_32Quads_top20_cov    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_cov.npz')

In [ ]:
CCMD_box_bottom20 = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
CCMD_box_top20    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')

CCMD_box_rsd_bot  = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_wprp_rsd_bottom20_pimax60_60_60_40.npz')
CCMD_box_rsd_top  = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_wprp_rsd_top20_pimax60_60_60_40.npz')

In [ ]:
CCMD_20Vars_box_rsd_bottom_mean = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_bottom_stat_mean.npz')
CCMD_20Vars_box_rsd_bottom_cov  = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_bottom_cov.npz'      )
CCMD_20Vars_box_rsd_top_mean    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_top_stat_mean.npz'   )
CCMD_20Vars_box_rsd_top_cov     = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_top_cov.npz'         )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(25, 15), dpi = 150)

# 创建主GridSpec：4行（每行包含原图+比值图），4列
gs_main = gridspec.GridSpec(4, 4, figure=fig, height_ratios=[5, 5, 5, 5], hspace=0.05, wspace=0)

ax_main = [[None for _ in range(4)] for _ in range(4)]  # 原图
ax_ratio = [[None for _ in range(4)] for _ in range(4)]  # 比值图

percent = 100

for j in range(4):
    for i in range(4):
        # 为每个单元格创建嵌套的GridSpec (2行1列，高度比例4:1)
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[j, i], 
                                                  height_ratios=[10, 5], hspace=0)
        ########################################################################################################################################
        # 原图（上部分）
        ax_main[j][i] = fig.add_subplot(gs_cell[0])
### bottom 20% #########################################################################################################################
        if j == 0: ### CCMD Main 4Quads
            stat = CCMD_Main_4Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
            cov  = CCMD_Main_4Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        ### CCMD 32Quads #############################################################################################################
        else:
            stat = CCMD_32Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
            cov  = CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(cov.diagonal())
        # cov_20Vars = CCMD_20Vars_box_rsd_bottom_cov['bottom_cov_'  + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        # yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
        yerr_total = np.sqrt((cov).diagonal())
        # positive value
        # mask = stat == stat
        ax_main[j][i].plot(rp_bins_x,  stat,  color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD')
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3)
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err
        ax_main[j][i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err  
        # negative value
        ax_main[j][i].plot(rp_bins_x, -stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
        ax_main[j][i].fill_between(rp_bins_x, -stat - yerr, -stat + yerr, color = 'forestgreen', alpha = 0.3, hatch = '///', facecolor = 'none')
        ax_main[j][i].fill_between(rp_bins_x, -(stat - yerr_total), -(stat - yerr), color = 'royalblue', 
                            hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
        ax_main[j][i].fill_between(rp_bins_x, -(stat + yerr), -(stat + yerr_total), color = 'royalblue', 
                            hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
        ### SDSS bright1 #############################################################################################################
        stat = SDSS_Mr_C_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(SDSS_cov_Mr_C_bottom20['wprp_bottom20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        # positive value
        mask = stat == stat
        ax_main[j][i].errorbar(rp_bins_x,  stat, yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 3, ms = 6, fmt = 'o')#, label = 'SDSS')
        # negative value
        ax_main[j][i].errorbar(rp_bins_x, -stat, yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 2.5, ms = 8, fmt = 'v')
### top 20% ############################################################################################################################
        if j == 0: ### CCMD Main 4Quads
            stat = CCMD_Main_4Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
            cov  = CCMD_Main_4Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        else:
        ### CCMD 32Quads ########################################################################################## 
            stat = CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
            cov  = CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(cov.diagonal())
        ax_main[j][i].plot(rp_bins_x, stat, color = 'forestgreen', alpha = 0.7, lw = 2, ls = '-', label = 'CCMD')
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.1)
        # cov_20Vars = CCMD_20Vars_box_rsd_top_cov[ 'top_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        # yerr_total = np.sqrt((cov + cov_20Vars).diagonal())    
        yerr_total = np.sqrt((cov).diagonal())    
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
        ax_main[j][i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
        ## SDSS ################################################################################################### 
        stat_SDSS = SDSS_Mr_C_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        yerr_SDSS = np.sqrt(SDSS_cov_Mr_C_top20['wprp_top20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        ax_main[j][i].errorbar(rp_bins_x, stat_SDSS, yerr = yerr_SDSS, color = 'orange', mfc = 'white', alpha = 0.7, 
                         capsize = 0, lw = 2, mew = 2, ms = 5, fmt = 'o', label = 'SDSS')

        ########################################################################################################################################
        # 比值图（下部分）
        ax_ratio[j][i] = fig.add_subplot(gs_cell[1])
        # 在比值图中绘图（替换为您的比值数据）
        # ratio_data = y / y.max()  # 示例数据
        # ax_ratio[j][i].plot(x, ratio_data, color='red', linewidth=1)
        ### CCMD
        ax_ratio[j][i].plot(rp_bins_x, (stat/stat-1)*percent, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-',)# label = 'CCMD')
        ax_ratio[j][i].fill_between(rp_bins_x, ((stat - yerr)/stat-1)*percent, ((stat + yerr)/stat-1)*percent, color = 'forestgreen', alpha = 0.3,)
        ax_ratio[j][i].fill_between(rp_bins_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) # add CCMD 20Vars box err
        ax_ratio[j][i].fill_between(rp_bins_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) # add CCMD 20Vars box err
        ### SDSS
        ax_ratio[j][i].errorbar(rp_bins_x, (stat_SDSS/stat-1)*percent, yerr = (yerr_SDSS/stat)*percent, 
                   color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                   marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white',)#  label = 'SDSS')
        ### ratio = 1
        ax_ratio[j][i].plot([-5, 60], [0, 0], color = 'gray', alpha = 0.6)
        ##########################################################################        
        # 调整主图四个边框的粗细
        for spine in ax_main[j][i].spines.values():
                # 对于主图 - 分别调整每个边框
                ax_main[j][i].spines['top'].set_linewidth(2)
                ax_main[j][i].spines['bottom'].set_linewidth(1) 
                ax_main[j][i].spines['left'].set_linewidth(2)
                ax_main[j][i].spines['right'].set_linewidth(2)
        for spine in ax_ratio[j][i].spines.values():
                # 对于比值图 - 分别调整每个边框
                ax_ratio[j][i].spines['top'].set_linewidth(1)
                ax_ratio[j][i].spines['bottom'].set_linewidth(2)
                ax_ratio[j][i].spines['left'].set_linewidth(2)
                ax_ratio[j][i].spines['right'].set_linewidth(2)

        ax_ratio[j][i].set_xscale('log')
        if j < 2:
            # ax_ratio[j][i].set_yticks([-30, 30], ['-30%', '30%'])
            # ax_ratio[j][i].set_ylim(-50, 50)
            ax_ratio[j][i].set_yticks([-50, 50], ['-50%', '50%'])
            ax_ratio[j][i].set_ylim(-110, 110)
        else:
            ax_ratio[j][i].set_yticks([-70, 70], ['-70%', '70%'])
            ax_ratio[j][i].set_ylim(-130, 130)
        ax_ratio[j][i].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_ratio[j][i].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_ratio[j][i].tick_params(axis='both', which='major', width=2, length=8, labelsize=24,)
        ax_ratio[j][i].tick_params(axis='both', which='minor', width=2, length=5, labelsize=24,)
        # ax_ratio[j][i].xaxis.set_minor_locator(plt.MultipleLocator(0.5))  # x轴子刻度
        # ax_ratio[j][i].yaxis.set_minor_locator(plt.MultipleLocator(0.5))  # y轴子刻度
        ax_ratio[j][i].set_xlim(0.15, 40)
        # ax_ratio[j][i].set_xlim(-0.9, 1.7)

        ax_main[j][i].loglog()
        # # ax_main[j][i].set_xscale('log')
        # ax_main[j][i].set_yticks([1, 3, 5], ['1', '3', '5'])
        ax_main[j][i].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_main[j][i].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_main[j][i].tick_params(axis='both', which='major', width=2, length=8, labelsize=24,)
        ax_main[j][i].tick_params(axis='both', which='minor', width=2, length=5, labelsize=24,)
        # ax_main[j][i].xaxis.set_minor_locator(plt.MultipleLocator(0.5))  # x轴子刻度
        # ax_main[j][i].yaxis.set_minor_locator(plt.MultipleLocator(1))  # y轴子刻度
        ax_main[j][i].set_xlim(0.15, 40)
        ax_main[j][i].set_ylim(0.5, 5000)
        # ax_main[j][i].set_xlim(-0.9, 1.7)
        # ax_main[j][i].set_ylim(0.1, 5.5)
            
for j in range(3):
    for i in [1, 2, 3]:
        ax_ratio[j][i].tick_params(axis='both', which='major', labelsize=24, labelbottom=False, labelleft=False)
        ax_ratio[j][i].tick_params(axis='both', which='minor', labelsize=24, labelbottom=False, labelleft=False)
        ax_ratio[j][0].tick_params(axis='both', which='major', labelsize=24, labelbottom=False)
        ax_ratio[j][0].tick_params(axis='both', which='minor', labelsize=24, labelbottom=False)
        ax_ratio[3][i].tick_params(axis='both', which='major', labelsize=24, labelleft=False)
        ax_ratio[3][i].tick_params(axis='both', which='minor', labelsize=24, labelleft=False)
     
for j in range(4):
    ax_main[j][0].text(0.2, 500, Mr_thr_text[j], fontsize=22, fontweight='bold',)  
    ax_main[0][0].text(10  , 120, 'top 20%', rotation=0, fontsize = 15, color = 'dimgray', alpha = 1)
    ax_main[0][0].text(10  , 2  , 'bot 20%', rotation=0, fontsize = 15, color = 'dimgray', alpha = 1)
    # ax_main[0][1].text(0.2, 500, 'CCMD Main 4Quads', rotation=0, fontsize = 20, color = 'black', alpha = 1)
#     ax_main[j][3].text(55, 15, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=24, fontweight='bold',)
#     ax_main[j][3].text(1.9, 1.5, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=24, fontweight='bold',)
    for i in [1, 2, 3]:
        ax_main[j][i].tick_params(axis='both', which='major', labelsize=24, labelbottom=False, labelleft=False)
        ax_main[j][i].tick_params(axis='both', which='minor', labelsize=24, labelbottom=False, labelleft=False)
    for i in range(4):
        ax_ratio[j][i].tick_params(axis = 'y', direction = 'in', which = u'both', labelsize = 18)
        ax_ratio[0][0].text(1, -65, 'Diff. (top 20%)', fontsize = 18, fontweight='normal', color = 'k')
         
legend = ax_main[3][0].legend(fontsize = 20, loc = 'upper right',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('royalblue')
legend.get_texts()[1].set_color('orange')

ax_main[0][0].set_title(r'$blue$'   , fontsize = 30, fontweight='medium', color = 'royalblue')
ax_main[0][1].set_title(r'$green$'  , fontsize = 30, fontweight='medium', color = 'forestgreen')
ax_main[0][2].set_title(r'$red$'    , fontsize = 30, fontweight='medium', color = 'red')
ax_main[0][3].set_title(r'$redder$' , fontsize = 30, fontweight='medium', color = 'red')

fig.supxlabel(r'$r_p~[h^{-1}{\rm Mpc}]$'  , fontsize=30, fontweight='bold', y = 0.04)
fig.supylabel(r'$w_p(r_p | \tilde{N}_g^8)$', fontsize=30, fontweight='bold', x = 0.07)

plt.tight_layout()
plt.show()